# Lecture 2: Python for Text Processing — Answer Key
### NLP Course 2027 — Week 01

---

This notebook provides complete, worked answers for the four exercises in **Lecture 2: Python for Text Processing**.

In [ ]:
import nltk

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
            'reuters', 'words']:
    nltk.download(pkg, quiet=True)

import re
import string
from collections import Counter
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords, reuters
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import FreqDist

print('Setup complete.')

---
## Exercise 2.1

**Task:** Write `word_char_stats(text)` returning a dict with: `total_chars`, `total_words`, `unique_words`, `avg_word_length`, `longest_word`.

**Key concept:** Distinguishing *tokens* (raw NLTK output) from *words* (alphabetic tokens only) is important — punctuation marks are tokens but not words. `set()` over lowercased words gives unique types (the vocabulary). Average word length reveals writing style; longer averages suggest academic or technical prose.

In [ ]:
from nltk.tokenize import word_tokenize

def word_char_stats(text):
    """
    Compute character and word statistics for the given text.

    Parameters
    ----------
    text : str
        Raw input string.

    Returns
    -------
    dict with keys:
        total_chars    – number of characters in the raw string
        total_words    – number of alphabetic tokens
        unique_words   – vocabulary size (case-insensitive)
        avg_word_length – mean length of alphabetic tokens
        longest_word   – the longest alphabetic token (first if tied)
    """
    tokens = word_tokenize(text)
    # Keep only purely alphabetic tokens — drops numbers and punctuation.
    words = [t for t in tokens if t.isalpha()]

    if not words:
        return {
            'total_chars': len(text),
            'total_words': 0,
            'unique_words': 0,
            'avg_word_length': 0.0,
            'longest_word': ''
        }

    # avg_word_length: mean character count across all word tokens.
    avg_len = sum(len(w) for w in words) / len(words)

    # longest_word: max by character length; lowercased for consistency.
    longest = max(words, key=len).lower()

    return {
        'total_chars': len(text),
        'total_words': len(words),
        'unique_words': len(set(w.lower() for w in words)),
        'avg_word_length': round(avg_len, 2),
        'longest_word': longest
    }


# ── Tests ────────────────────────────────────────────────────────────────────
samples = [
    "Natural Language Processing is a fascinating field of computer science.",
    "To be, or not to be, that is the question.",
    "The quick brown fox jumps over the lazy dog.",
]

for s in samples:
    stats = word_char_stats(s)
    print(f'Text: "{s[:45]}..."')
    for k, v in stats.items():
        print(f'  {k:<18}: {v}')
    print()

**Notes on the implementation:**

- `t.isalpha()` is the correct filter — `t.isalnum()` would include tokens like `'1st'` which mix letters and digits.
- The guard for an empty `words` list prevents division-by-zero on empty or punctuation-only input.
- `max(words, key=len)` is O(n) — more efficient than sorting the whole list just to get one word.
- Lowercasing in `set(w.lower() for w in words)` ensures `'The'` and `'the'` are counted as the same type.

---
## Exercise 2.2

**Task:** Compare tokenization of `"I can't believe it's 5 o'clock!"` using `str.split()`, `word_tokenize()`, and `RegexpTokenizer(r'\w+')`. Which handles contractions best?

**Key concept:** Tokenization is not trivial. The three approaches represent a spectrum from naïve (split) to rule-based (regex) to linguistically-informed (NLTK). Contractions like `can't` are a classic challenge: they are one orthographic word but two morpho-syntactic units (`can` + `not`).

In [ ]:
from nltk.tokenize import word_tokenize, RegexpTokenizer

text = "I can't believe it's 5 o'clock!"

# Approach 1: naive whitespace split
split_tokens = text.split()

# Approach 2: NLTK word_tokenize — trained on Penn Treebank conventions
nltk_tokens = word_tokenize(text)

# Approach 3: regex \w+ — matches sequences of [a-zA-Z0-9_], strips all punct
re_tok = RegexpTokenizer(r'\w+')
regex_tokens = re_tok.tokenize(text)

print(f'Original text : {text}')
print()
print(f'split()        ({len(split_tokens):2d} tokens): {split_tokens}')
print(f'word_tokenize  ({len(nltk_tokens):2d} tokens): {nltk_tokens}')
print(f'RegexpTokenizer({len(regex_tokens):2d} tokens): {regex_tokens}')

# --- Analysis table ---
print()
print('Analysis:')
analysis = [
    ('split()', 'Keeps punct attached: "can\'t" stays as one token. No expansion.', 'Poor'),
    ('word_tokenize', 'Splits "can\'t" → ["ca", "n\'t"], "it\'s" → ["it", "'\'s"]. Follows Penn Treebank.', 'Best for NLP pipelines'),
    ('RegexpTokenizer(\\w+)', 'Strips all apostrophes: "can\'t" → ["can", "t"], losing morphology.', 'Loses contraction structure'),
]
for name, desc, verdict in analysis:
    print(f'  {name:<22}: {verdict}')
    print(f'    {desc}')

**Conclusion:**

| Tokenizer | `can't` result | Verdict |
|-----------|---------------|--------|
| `split()` | `can't` | Attached punctuation, no expansion |
| `word_tokenize` | `ca`, `n't` | Penn Treebank convention — best for POS tagging |
| `RegexpTokenizer(r'\w+')` | `can`, `t` | Destroys the contraction; `t` is meaningless |

`word_tokenize` handles contractions best *for downstream NLP tasks* (tagging, parsing) because it produces linguistically meaningful units. The `\w+` tokenizer is useful when you explicitly want to discard punctuation and contractions (e.g., bag-of-words classification where you just need word stems).

---
## Exercise 2.3

**Task:** Load the Reuters corpus and find the 20 most common *content words* (non-stopwords, alphabetic) across all documents.

**Key concept:** The Reuters corpus contains ~10,800 financial news articles. Content-word frequency reveals the dominant topics of a corpus. Removing stopwords is a standard preprocessing step that dramatically shifts the frequency ranking from function words (`the`, `a`, `of`) to meaningful domain terms.

In [ ]:
from nltk.corpus import reuters, stopwords
from nltk import FreqDist

stop_words = set(stopwords.words('english'))

print('Loading Reuters corpus ...')
print(f'  Documents : {len(reuters.fileids()):,}')
print(f'  Categories: {len(reuters.categories())}')
print()

# Stream all words through filters — avoids building a huge intermediate list.
content_words = [
    w.lower()
    for w in reuters.words()          # iterate all words in the corpus
    if w.isalpha()                     # drop numbers, punctuation, symbols
    and w.lower() not in stop_words   # drop function words
    and len(w) > 2                    # drop very short fragments ('lt', 'vs')
]

fdist = FreqDist(content_words)

print(f'Total content-word tokens : {len(content_words):,}')
print(f'Unique content-word types : {fdist.N():,}  (vocab size: {len(fdist):,})')
print()
print('Top 20 content words in Reuters:')
print(f'{"Rank":<6} {"Word":<20} {"Count":>8} {"% of content tokens":>20}')
print('-' * 60)
for rank, (word, count) in enumerate(fdist.most_common(20), 1):
    pct = 100 * count / len(content_words)
    print(f'{rank:<6} {word:<20} {count:>8,} {pct:>19.2f}%')

**Expected observations:**

The top content words will include financial/business terms like `said`, `year`, `billion`, `company`, `trade`, `government`, `bank` — all strongly characteristic of financial news. This confirms that frequency analysis is a quick, zero-shot way to characterise what a corpus is *about*.

`said` ranking very high is common in news corpora because journalists attribute every statement to a source. This is sometimes called the **attribution verb** pattern.

---
## Exercise 2.4

**Task:** Build a `TextPreprocessor` class with `__init__` parameters `(lowercase, remove_punct, remove_stopwords, stem)` and a `process(text)` method.

**Key concept:** Wrapping a preprocessing pipeline in a class makes the configuration explicit and reusable — the same object can preprocess thousands of documents consistently. Each boolean flag controls one independent transformation stage.

In [ ]:
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


class TextPreprocessor:
    """
    Configurable text preprocessing pipeline.

    Parameters
    ----------
    lowercase        : bool  – convert all tokens to lowercase
    remove_punct     : bool  – drop punctuation tokens
    remove_stopwords : bool  – drop NLTK English stopwords
    stem             : bool  – apply Porter stemming to every remaining token
    """

    def __init__(self,
                 lowercase: bool = True,
                 remove_punct: bool = True,
                 remove_stopwords: bool = True,
                 stem: bool = False):
        self.lowercase = lowercase
        self.remove_punct = remove_punct
        self.remove_stopwords = remove_stopwords
        self.stem = stem

        # Initialise heavy resources once at construction time, not per call.
        self._punct_set = set(string.punctuation)
        self._stop_words = set(stopwords.words('english')) if remove_stopwords else set()
        self._stemmer = PorterStemmer() if stem else None

    def process(self, text: str) -> list:
        """
        Tokenise and apply the configured transformations.

        Returns
        -------
        list of str  – processed tokens
        """
        # Step 1: tokenise
        tokens = word_tokenize(text)

        # Step 2: lowercase
        if self.lowercase:
            tokens = [t.lower() for t in tokens]

        # Step 3: remove punctuation tokens
        if self.remove_punct:
            tokens = [t for t in tokens if t not in self._punct_set and t.isalpha()]

        # Step 4: remove stopwords
        if self.remove_stopwords:
            tokens = [t for t in tokens if t not in self._stop_words]

        # Step 5: stem
        if self.stem:
            tokens = [self._stemmer.stem(t) for t in tokens]

        return tokens

    def __repr__(self):
        return (f'TextPreprocessor(lowercase={self.lowercase}, '
                f'remove_punct={self.remove_punct}, '
                f'remove_stopwords={self.remove_stopwords}, '
                f'stem={self.stem})')


# ── Demo ─────────────────────────────────────────────────────────────────────
sample = "The researchers are studying 3 NLP models and evaluating their performance!"

configs = [
    dict(lowercase=True,  remove_punct=True,  remove_stopwords=False, stem=False),
    dict(lowercase=True,  remove_punct=True,  remove_stopwords=True,  stem=False),
    dict(lowercase=True,  remove_punct=True,  remove_stopwords=True,  stem=True),
]

print(f'Input: "{sample}"\n')
for cfg in configs:
    pp = TextPreprocessor(**cfg)
    result = pp.process(sample)
    label = ('no stops, no stem' if not cfg['remove_stopwords'] and not cfg['stem']
             else 'stops removed, no stem' if cfg['remove_stopwords'] and not cfg['stem']
             else 'stops removed + stem')
    print(f'[{label}]')
    print(f'  {result}\n')

**Design notes:**

- The `PorterStemmer` and `stopwords` are loaded *once* in `__init__`, not on every call to `process()`. This matters when processing thousands of documents — object initialisation is expensive.
- `t not in self._punct_set and t.isalpha()` combines two checks: the set lookup catches single-character punctuation tokens, while `isalpha()` catches tokens like `--` or `...` that are not in `string.punctuation`.
- Setting `stem=True` but `remove_stopwords=False` is valid — the class makes no assumptions about configuration order.

**Extension:** Add `lemmatize=True` parameter using `WordNetLemmatizer` as a mutually-exclusive alternative to stemming.

---
## Summary of Key Concepts

| Exercise | Concept | Key Takeaway |
|----------|---------|-------------|
| 2.1 | Text statistics | Distinguish tokens vs words; `max(..., key=len)` not sort |
| 2.2 | Tokenization comparison | `word_tokenize` best for NLP; `\w+` erases contraction meaning |
| 2.3 | Corpus frequency analysis | Stopword removal reveals domain vocabulary (Reuters → finance) |
| 2.4 | Preprocessing pipeline class | Load heavy resources once in `__init__`; boolean flags for reproducibility |

---
*NLP Course 2027 — Week 01*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**